# 02 — Candidate reduction and multivariate protein shortlist

The original dataset contained **2,944 protein variables**, which was too large for an exhaustive combination search.

A practical laboratory constraint motivated reducing this space: measuring a small number of representative proteins is easier than using a very large panel, especially when multiple proteins may reflect similar molecular pathways.

Two independent **AI-assisted pathway grouping passes** were used only as a coarse dimensionality-reduction heuristic:

- one grouping produced **8 broad pathway groups**; the 3 strongest individual candidates from each group were retained;
- a second grouping produced **34 finer groups**; the strongest individual candidate from each group was retained;
- both lists were merged and duplicates removed, leaving roughly **50 candidate proteins**.

The AI groupings were known to be imperfect and were **not treated as biological ground truth**. Their purpose was only to eliminate a large number of low-priority variables before multivariate modelling.

The full pathway-assignment tables are not published because they expose the confidential dataset's complete protein-variable inventory, and those intermediate files were not retained after the hackathon.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

TRAIN_FILENAME = "TrainData.csv"
TARGET = "clinic_group"

CANDIDATE_PROTEINS = ['NPX_AREG', 'NPX_VCAM1', 'NPX_CCL18', 'NPX_ITGB6', 'NPX_PTX3', 'NPX_STAT84L1', 'NPX_CHI3L1', 'NPX_HLA477L1', 'NPX_FGA2242', 'NPX_KIAA3833', 'NPX_TF370S', 'NPX_LOC407R1', 'NPX_CD44R', 'NPX_JNK11R1', 'NPX_NOTCH36L', 'NPX_IgG4141', 'NPX_APOB436A', 'NPX_NF110', 'NPX_FAM332L', 'NPX_GENE396R', 'NPX_LINC53L', 'NPX_RP14S', 'NPX_TCR97L', 'NPX_PROT1792', 'NPX_AKT642', 'NPX_AP331L1', 'NPX_PCSK311A', 'NPX_WNT405A', 'NPX_TNF145R1', 'NPX_C346L2', 'NPX_VEGF37A', 'NPX_BMP358A', 'NPX_WNT4411', 'NPX_STAT266L2', 'NPX_AKT1441', 'NPX_AKT4152', 'NPX_IL484L1', 'NPX_CCL3543', 'NPX_VTN469A', 'NPX_FBN251R1', 'NPX_IgA87P', 'NPX_HLA462L1', 'NPX_IgM4601', 'NPX_APOC137A', 'NPX_APOC93', 'NPX_LPA168L', 'NPX_HP114R', 'NPX_HP234A', 'NPX_ALB489', 'NPX_WFDC2']

def resolve_data_file(filename):
    for path in [Path("data") / filename, Path(filename)]:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find {filename}. Put the authorized file in ./data/ "
        "or next to this notebook. Do not commit it to GitHub."
    )

train = pd.read_csv(resolve_data_file(TRAIN_FILENAME))
train.columns = train.columns.astype(str).str.strip()

available_candidates = [p for p in CANDIDATE_PROTEINS if p in train.columns]

print(f"Surviving hardcoded candidate list: {len(CANDIDATE_PROTEINS)} proteins")
print(f"Candidates present in TrainData: {len(available_candidates)}")


Surviving hardcoded candidate list: 50 proteins
Candidates present in TrainData: 50


## Multivariate shortlist

After the pathway-assisted prefilter, the candidate proteins were assessed in multivariate modelling rather than relying only on their individual behaviour.

The exact intermediate script that generated the final compact-panel table was not retained. The saved aggregate results do preserve the method and the panels that were compared:

- L2-regularized logistic regression;
- stratified 5-fold cross-validation;
- numeric imputation and standardization;
- categorical imputation and one-hot encoding;
- compact protein panels compared together with clinical information.

The table below reproduces the retained aggregate Step 30 results used during the hackathon. These are aggregate model results, not participant-level data.

In [2]:
RecoveredPanels = pd.DataFrame([
    {
        "Panel": "Stable_core_5",
        "N_Proteins": 5,
        "Mean_CV_AUC": 0.9612,
        "SD_CV_AUC": 0.0162,
        "Gain_vs_Clinical": 0.0179,
        "Protein_Members": "GDF15; SFTPB; MMP7; FGA2242; VCAM1",
    },
    {
        "Panel": "Stable_core_4",
        "N_Proteins": 4,
        "Mean_CV_AUC": 0.9597,
        "SD_CV_AUC": 0.0170,
        "Gain_vs_Clinical": 0.0164,
        "Protein_Members": "GDF15; SFTPB; MMP7; FGA2242",
    },
    {
        "Panel": "GDF15_SFTPB_MMP7",
        "N_Proteins": 3,
        "Mean_CV_AUC": 0.9590,
        "SD_CV_AUC": 0.0189,
        "Gain_vs_Clinical": 0.0157,
        "Protein_Members": "GDF15; SFTPB; MMP7",
    },
    {
        "Panel": "GDF15_SFTPB_FGA2242",
        "N_Proteins": 3,
        "Mean_CV_AUC": 0.9588,
        "SD_CV_AUC": 0.0188,
        "Gain_vs_Clinical": 0.0155,
        "Protein_Members": "GDF15; SFTPB; FGA2242",
    },
    {
        "Panel": "Earlier_5",
        "N_Proteins": 5,
        "Mean_CV_AUC": 0.9582,
        "SD_CV_AUC": 0.0225,
        "Gain_vs_Clinical": 0.0149,
        "Protein_Members": "GDF15; SFTPB; MMP7; WFDC2; AREG",
    },
    {
        "Panel": "GDF15_SFTPB",
        "N_Proteins": 2,
        "Mean_CV_AUC": 0.9579,
        "SD_CV_AUC": 0.0223,
        "Gain_vs_Clinical": 0.0146,
        "Protein_Members": "GDF15; SFTPB",
    },
    {
        "Panel": "GDF15",
        "N_Proteins": 1,
        "Mean_CV_AUC": 0.9522,
        "SD_CV_AUC": 0.0185,
        "Gain_vs_Clinical": 0.0089,
        "Protein_Members": "GDF15",
    },
    {
        "Panel": "Clinical_only",
        "N_Proteins": 0,
        "Mean_CV_AUC": 0.9433,
        "SD_CV_AUC": 0.0159,
        "Gain_vs_Clinical": 0.0000,
        "Protein_Members": "",
    },
])

display(RecoveredPanels)


,Panel,N_Proteins,Mean_CV_AUC,SD_CV_AUC,Gain_vs_Clinical,Protein_Members
0,Stable_core_5,5,0.9612,0.0162,0.0179,GDF15; SFTPB; MMP7; FGA2242; VCAM1
1,Stable_core_4,4,0.9597,0.0170,0.0164,GDF15; SFTPB; MMP7; FGA2242
2,GDF15_SFTPB_MMP7,3,0.9590,0.0189,0.0157,GDF15; SFTPB; MMP7
3,GDF15_SFTPB_FGA2242,3,0.9588,0.0188,0.0155,GDF15; SFTPB; FGA2242
4,Earlier_5,5,0.9582,0.0225,0.0149,GDF15; SFTPB; MMP7; WFDC2; AREG
5,GDF15_SFTPB,2,0.9579,0.0223,0.0146,GDF15; SFTPB
6,GDF15,1,0.9522,0.0185,0.0089,GDF15
7,Clinical_only,0,0.9433,0.0159,0.0000,


In [3]:
protein_union = set()

for members in RecoveredPanels["Protein_Members"]:
    if members:
        protein_union.update(p.strip() for p in members.split(";"))

FINAL_7_PROTEINS = [
    "GDF15",
    "SFTPB",
    "MMP7",
    "FGA2242",
    "VCAM1",
    "WFDC2",
    "AREG",
]

assert set(FINAL_7_PROTEINS) == protein_union

FinalShortlist = pd.DataFrame({
    "Protein": [f"NPX_{p}" for p in FINAL_7_PROTEINS]
})

display(FinalShortlist)


,Protein
0,NPX_GDF15
1,NPX_SFTPB
2,NPX_MMP7
3,NPX_FGA2242
4,NPX_VCAM1
5,NPX_WFDC2
6,NPX_AREG


## Output of this stage

The union of the strongest compact panels produced **7 proteins**:

**GDF15, SFTPB, MMP7, FGA2242, VCAM1, WFDC2 and AREG.**

These seven proteins are carried forward to the binary classification analysis. Notebook 03 evaluates them individually; Notebook 04 tests every possible non-empty combination of them.